# Initial EDA - Customer Info

This notebook starts the initial exploratory data analysis for the customer segmentation project.

This phase focuses only on `customer_info`, the customer-level dataset that will be the base for segmentation because every customer must be included in the final clustering output.

## Imports and Data Loading

Load the raw CSV files directly with `pd.read_csv` from `../data/raw/`. No working-directory changes or fallback paths are used.

In [ ]:
import ast

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
customer_info = pd.read_csv("../data/raw/customer_info.csv")
customer_info.head()

## Basic Structure

Check the size, columns, data types, and summary statistics before making any assumptions about the data.

In [ ]:
print(f"Rows: {customer_info.shape[0]:,}")
print(f"Columns: {customer_info.shape[1]:,}")

customer_info.columns.to_frame(index=False, name="column")

In [ ]:
customer_info.dtypes.to_frame(name="dtype")

In [ ]:
customer_info.info()

In [ ]:
customer_info.describe()

In [ ]:
customer_info.select_dtypes(include=["object", "string"]).describe()

## Key Integrity Checks

`customer_id` must be reliable because the final output needs one assigned cluster for every customer.

In [ ]:
customer_id_checks = pd.DataFrame(
    {
        "check": [
            "duplicated customer_id count",
            "unique customer_id count",
            "customer_id is unique",
        ],
        "value": [
            customer_info["customer_id"].duplicated().sum(),
            customer_info["customer_id"].nunique(dropna=False),
            customer_info["customer_id"].is_unique,
        ],
    }
)

customer_id_checks

## Missing Values

Review missingness by column before deciding how to handle any values later.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": customer_info.isna().sum(),
        "missing_percentage": customer_info.isna().mean() * 100,
    }
).sort_values("missing_percentage", ascending=False)

missing_summary

In [ ]:
missing_with_values = missing_summary[missing_summary["missing_count"] > 0].reset_index(names="column")

if missing_with_values.empty:
    print("No missing values found.")
else:
    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=missing_with_values,
        x="missing_percentage",
        y="column",
        color="steelblue",
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Missing percentage")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()

Missingness is concentrated in `loyalty_card_number`, while several customer behavior fields have smaller missing shares. These columns should be handled carefully later, but no imputation or preprocessing is done in this notebook.

## Numerical Distributions

Plot the main numerical fields that may matter for customer segmentation. This is only inspection, not feature engineering.

In [ ]:
lifetime_spend_columns = [column for column in customer_info.columns if column.startswith("lifetime_spend_")]

numerical_columns = [
    "kids_home",
    "teens_home",
    "number_complaints",
    "distinct_stores_visited",
    *lifetime_spend_columns,
    "lifetime_total_distinct_products",
    "year_first_transaction",
    "percentage_of_products_bought_promotion",
    "typical_hour",
]
numerical_columns = [column for column in dict.fromkeys(numerical_columns) if column in customer_info.columns]

customer_info[numerical_columns].hist(bins=30, figsize=(16, 20), color="steelblue", edgecolor="white")
plt.suptitle("Numerical Distributions", y=1.01)
plt.tight_layout()
plt.show()

Spend columns appear right-skewed, which is common for customer purchasing data. Range checks below are more important at this stage than reshaping distributions.

## Categorical Distributions

Inspect simple categorical fields and visible name prefixes.

In [ ]:
customer_info["customer_gender"].value_counts(dropna=False)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=customer_info, x="customer_gender", color="steelblue")
plt.title("Customer Gender Counts")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
customer_info["loyalty_card_number"].value_counts(dropna=False)

In [ ]:
loyalty_status = customer_info["loyalty_card_number"].map({1.0: "Present"}).fillna("Missing")
loyalty_status_counts = loyalty_status.value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=loyalty_status_counts.index, y=loyalty_status_counts.values, color="steelblue")
plt.title("Loyalty Card Number Presence")
plt.xlabel("Loyalty card number")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
name_prefix_counts = (
    customer_info["customer_name"]
    .astype("string")
    .str.extract(r"^(Bsc\.|Msc\.|Phd\.)", expand=False)
    .fillna("No visible prefix")
    .value_counts()
)

name_prefix_counts

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(x=name_prefix_counts.index, y=name_prefix_counts.values, color="steelblue")
plt.title("Visible Customer Name Prefixes")
plt.xlabel("Prefix")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Gender is balanced. `loyalty_card_number` looks more like an indicator than a meaningful numeric value, and the name prefix inspection is only a possible clue for later feature engineering.

## Data Quality Checks

Check for suspicious ranges without changing the data.

In [ ]:
promotion = customer_info["percentage_of_products_bought_promotion"]
year_first_transaction = customer_info["year_first_transaction"]
typical_hour = customer_info["typical_hour"]

range_checks = pd.DataFrame(
    {
        "check": [
            "promotion percentage outside [0, 1]",
            "year_first_transaction after 2026",
            "typical_hour outside [0, 23]",
            "negative kids_home",
            "negative teens_home",
            "negative number_complaints",
            "negative distinct_stores_visited",
        ],
        "count": [
            ((promotion < 0) | (promotion > 1)).sum(),
            (year_first_transaction > 2026).sum(),
            ((typical_hour < 0) | (typical_hour > 23)).sum(),
            (customer_info["kids_home"] < 0).sum(),
            (customer_info["teens_home"] < 0).sum(),
            (customer_info["number_complaints"] < 0).sum(),
            (customer_info["distinct_stores_visited"] < 0).sum(),
        ],
    }
)

range_checks

In [ ]:
spend_and_count_columns = lifetime_spend_columns + ["lifetime_total_distinct_products"]
negative_value_counts = customer_info[spend_and_count_columns].lt(0).sum().sort_values(ascending=False)

negative_value_counts[negative_value_counts > 0]

## Data Quality Notes

- Missing values are present, especially in `loyalty_card_number`; smaller missing shares also appear in several behavior and spend fields.
- Possible invalid values should be reviewed before preprocessing, especially promotion percentages outside `[0, 1]`.
- `loyalty_card_number` may be better treated as a loyalty flag later because the visible non-missing value is `1.0`.
- `customer_name` should not be used directly for modelling.
- Degree prefixes such as `Bsc.`, `Msc.`, and `Phd.` may become a useful engineered feature later, but no feature is created here.
- `year_first_transaction` includes suspicious future-looking values after 2026 that should be validated.
- `percentage_of_products_bought_promotion` includes invalid negative values and should be reviewed before modelling.

# Customer Basket Initial EDA

This section reviews the transaction-level `customer_basket` dataset. It keeps the work to initial inspection only: no preprocessing, modelling, or association rules.

## Customer Basket Data Loading

Load the raw CSV directly from the project data folder.

In [ ]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_basket.head()

## Basic Structure

Check the size, columns, data types, and summary statistics before making assumptions about basket-level data.

In [ ]:
print(f"Rows: {customer_basket.shape[0]:,}")
print(f"Columns: {customer_basket.shape[1]:,}")

customer_basket.columns.to_frame(index=False, name="column")

In [ ]:
customer_basket.dtypes.to_frame(name="dtype")

In [ ]:
customer_basket.info()

In [ ]:
customer_basket.head()

In [ ]:
customer_basket.describe(include="all")

## Integrity Checks

Check invoice and row uniqueness before using baskets for customer-level analysis.

In [ ]:
basket_integrity_checks = pd.DataFrame(
    {
        "check": [
            "duplicated invoice_id count",
            "unique invoice_id count",
            "invoice_id is unique",
            "unique customer_id count",
            "duplicated full rows count",
        ],
        "value": [
            customer_basket["invoice_id"].duplicated().sum(),
            customer_basket["invoice_id"].nunique(dropna=False),
            customer_basket["invoice_id"].is_unique,
            customer_basket["customer_id"].nunique(dropna=False),
            customer_basket.duplicated().sum(),
        ],
    }
)

basket_integrity_checks

## Missing Values

Review missingness by column before deciding whether any cleaning will be needed later.

In [ ]:
basket_missing_summary = pd.DataFrame(
    {
        "missing_count": customer_basket.isna().sum(),
        "missing_percentage": customer_basket.isna().mean() * 100,
    }
).sort_values("missing_percentage", ascending=False)

basket_missing_summary

## Parse List of Goods

Parse `list_of_goods` into a temporary analysis column while preserving the original raw string column.

In [ ]:
customer_basket["goods_list"] = customer_basket["list_of_goods"].apply(ast.literal_eval)
customer_basket[["list_of_goods", "goods_list"]].head()

## Basket Size Analysis

Count the number of products in each parsed basket for initial distribution checks.

In [ ]:
customer_basket["basket_size"] = customer_basket["goods_list"].str.len()
customer_basket["basket_size"].describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(customer_basket["basket_size"], discrete=True, color="steelblue")
plt.title("Basket Size Distribution")
plt.xlabel("Number of products in basket")
plt.ylabel("Number of baskets")
plt.tight_layout()
plt.show()

Basket sizes range from 1 to 18 products and are centered around 9 products, which looks reasonable for initial basket-level analysis.

## Product Frequency Analysis

Explode the parsed lists to inspect individual product frequency across all baskets.

In [ ]:
basket_products = customer_basket["goods_list"].explode()
print(f"Total unique products: {basket_products.nunique():,}")

In [ ]:
top_20_products = basket_products.value_counts().head(20)
top_20_products

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(x=top_20_products.values, y=top_20_products.index, color="steelblue")
plt.title("Top 20 Most Frequent Products")
plt.xlabel("Count")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

There are 164 unique products. A few products appear much more often than others, especially `asparagus` and `airpods`, so product imbalance should be kept in mind later.

## Customer Transaction Frequency

Count how many baskets appear for each customer in `customer_basket`.

In [ ]:
baskets_per_customer = customer_basket["customer_id"].value_counts()
baskets_per_customer.describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(baskets_per_customer, discrete=True, color="steelblue")
plt.title("Baskets per Customer Distribution")
plt.xlabel("Number of baskets")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

In [ ]:
top_10_customers_by_baskets = baskets_per_customer.head(10).rename_axis("customer_id").reset_index(name="basket_count")
top_10_customers_by_baskets

Most customers have only a few baskets in this dataset, while a small number of customers have much higher transaction counts.

## Customer Basket Data Quality Notes

- `invoice_id` appears unique in `customer_basket`; no duplicate invoice IDs were found.
- `customer_basket` has no missing values in the available columns.
- `list_of_goods` parses successfully into Python lists using `ast.literal_eval`, while the original raw column is preserved.
- Basket sizes look reasonable for initial inspection, ranging from 1 to 18 products.
- Some products dominate the transaction counts, with `asparagus` and `airpods` clearly leading the top-frequency table.
- Customer-level integration with `customer_info` will be checked in the next phase.

# Dataset Relationship Validation

Validate how `customer_info` and `customer_basket` connect through `customer_id`. This remains initial EDA only; no feature engineering dataset is saved.

## Customer ID Set Checks

Compare the customer ID sets to confirm whether basket records can be linked back to the customer-level table.

In [ ]:
customer_info_ids = set(customer_info["customer_id"])
customer_basket_ids = set(customer_basket["customer_id"])

customer_ids_in_both = customer_info_ids.intersection(customer_basket_ids)
customer_ids_basket_not_info = customer_basket_ids.difference(customer_info_ids)
customer_ids_info_not_basket = customer_info_ids.difference(customer_basket_ids)

customer_id_relationship_checks = pd.DataFrame(
    {
        "check": [
            "unique customer_id in customer_info",
            "unique customer_id in customer_basket",
            "customer_id present in both datasets",
            "customer_id in customer_basket but not customer_info",
            "customer_id in customer_info but not customer_basket",
        ],
        "value": [
            len(customer_info_ids),
            len(customer_basket_ids),
            len(customer_ids_in_both),
            len(customer_ids_basket_not_info),
            len(customer_ids_info_not_basket),
        ],
    }
)

customer_id_relationship_checks

## Integrity Conclusion

Confirm whether every transaction-level customer ID exists in the customer-level dataset.

In [ ]:
all_basket_customers_in_info = len(customer_ids_basket_not_info) == 0
print(f"Every customer_id in customer_basket exists in customer_info: {all_basket_customers_in_info}")

Every `customer_id` in `customer_basket` exists in `customer_info`, so basket records can be safely linked to customer records. This matters because future customer-level analysis must not introduce transactions that cannot be assigned back to a known customer.

## Coverage Analysis

Measure how many customers in `customer_info` have at least one basket record.

In [ ]:
customer_basket_coverage = pd.DataFrame(
    {
        "coverage_status": ["customers_with_basket", "customers_without_basket"],
        "count": [len(customer_ids_in_both), len(customer_ids_info_not_basket)],
    }
)
customer_basket_coverage["percentage"] = (
    customer_basket_coverage["count"] / len(customer_info_ids) * 100
)

customer_basket_coverage

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=customer_basket_coverage, x="coverage_status", y="count", color="steelblue")
plt.title("Customer Basket Coverage")
plt.xlabel("Coverage status")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

Most customers in `customer_info` have at least one basket record, but a meaningful minority do not. Those customers still need to remain in the final customer-level dataset.

## Transaction Count Merge for Analysis Only

Create a temporary basket count summary and merge it into `customer_info` only for inspection.

In [ ]:
basket_counts = (
    customer_basket
    .groupby("customer_id")
    .size()
    .reset_index(name="basket_count")
)

customer_info_with_basket_counts = customer_info.merge(basket_counts, on="customer_id", how="left")
customer_info_with_basket_counts["basket_count"] = (
    customer_info_with_basket_counts["basket_count"].fillna(0).astype(int)
)

customer_info_with_basket_counts[["customer_id", "basket_count"]].head()

In [ ]:
customer_info_with_basket_counts["basket_count"].describe()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(customer_info_with_basket_counts["basket_count"], discrete=True, color="steelblue")
plt.title("Basket Count Distribution Across Customer Info")
plt.xlabel("Number of baskets")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()

## Relationship Data Quality Notes

- `customer_basket` can be safely linked to `customer_info` because every basket customer ID appears in the customer-level table.
- Some `customer_info` customers do not have basket records; in this data, 4,911 customers have no matching basket.
- Final clustering must still include every `customer_id` from `customer_info`, including customers without transaction records.
- Basket-derived features should be handled carefully later because not every customer has basket data.
- The next phase will define data quality decisions before feature engineering.

# Data Quality Decisions Before Feature Engineering

This section records the main data quality decisions that will guide feature engineering. No preprocessing, modelling, or saved feature table is created here.

## Missing Values

- Missing values will not be dropped blindly.
- Treatment will depend on the meaning of each column.
- Numerical spend and behavior features will likely use median values, or zero where zero has clear business meaning.
- Basket-derived missing values for customers without baskets should become `0` for count-based features later.

## `loyalty_card_number`

- `loyalty_card_number` should not be used as a raw number.
- It should become a binary `has_loyalty_card` feature later.
- Missing values likely indicate no loyalty card, unless later evidence suggests otherwise.

## `customer_name`

- `customer_name` should not be used directly for modelling.
- It may contain academic prefixes such as `Bsc.`, `Msc.`, and `Phd.`.
- Prefixes may be extracted into a simple `degree_level` feature later if useful.

## `year_first_transaction`

- `year_first_transaction` should not be used raw without caution.
- It should likely become customer tenure.
- Suspicious or future-looking years should be checked and handled during feature engineering or preprocessing.

## `percentage_of_products_bought_promotion`

- `percentage_of_products_bought_promotion` should be checked for invalid values outside a valid percentage range.
- Invalid values should be handled before modelling.
- It may become an important proxy for promotion sensitivity.

## Location Columns

- `latitude` and `longitude` should be considered carefully.
- They may help geographic segmentation, but can also dominate distance-based clustering if not scaled properly.
- For a first modelling baseline, they may be excluded or tested separately.

## Basket Coverage

- `customer_basket` links safely to `customer_info` because all basket customer IDs exist in the customer-level table.
- Some `customer_info` customers have no basket records.
- Basket-derived features must be created carefully so every customer remains in the final dataset.
- Count-based basket features can be filled with `0` for customers without basket records.

## Modelling Implication

- The base clustering dataset should start from `customer_info`.
- `customer_basket` should enrich the analysis, cluster profiling, and promotions.
- The final output must include every `customer_id`.

The next phase is feature engineering, where these decisions will be implemented into a customer-level feature table.